In [1]:
!pip install markovianbandit-pkg


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import markovianbandit as bandit

In [3]:
import numpy as np
import pandas as pd

In [5]:
NUM_MACHINES = 3
NUM_DAMAGE_STATES = 8  # states (0: pristine, 1-6: damage, 7: breakdown)
D_values = [0, 10, 20, 25, 30, 40, 50]
a_values = [1.5, 2.0, 3.0, 4.0, 5.0]
DISCOUNT_FACTOR = 0.95

In [6]:
def whittle_indices(P0, P1, R0, R1):
  model = bandit.restless_bandit_from_P0P1_R0R1(P0, P1, R0, R1)
  whittle_indices = model.whittle_indices(discount = 0.95)
  return whittle_indices

In [7]:
def whittle_indices_undiscounted(P0, P1, R0, R1):
  model = bandit.restless_bandit_from_P0P1_R0R1(P0, P1, R0, R1)
  whittle_indices = model.whittle_indices()
  return whittle_indices

In [8]:
P0 = [[0.17900487, 0.82099513, 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.41451341, 0.        , 0.58548659, 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.65119585, 0.        , 0.        , 0.34880415, 0.        ,
         0.        , 0.        , 0.        ],
        [0.73231312, 0.        , 0.        , 0.        , 0.26768688,
         0.        , 0.        , 0.        ],
        [0.7712476 , 0.        , 0.        , 0.        , 0.        ,
         0.2287524 , 0.        , 0.        ],
        [0.79930418, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.20069582, 0.        ],
        [0.83813715, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.16186285],
        [1.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ]]

P1 = [[1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0],
      [1, 0, 0, 0, 0, 0, 0, 0]]

R0 = [0,  -13.98845706, -15.0964306,  -26.83213291, -31.49320303,
 -58.69191633, -59.90438515, -61.42409872]

R1 = [-26.82154652, -26.82154652, -26.82154652, -26.82154652, -26.82154652,
 -26.82154652, -26.82154652, -26.82154652]

In [9]:
len(R0)

8

In [10]:
indices = whittle_indices(P0, P1, R0, R1)

In [11]:
indices

array([-26.82154652,  -7.18837825,  -8.27179762,   5.95393089,
        10.78421911,  42.98422279,  41.05516858,  34.6025522 ])

In [12]:
indices = whittle_indices_undiscounted(P0, P1, R0, R1)

In [13]:
indices

array([-26.82154652,  -7.00314049,  -8.14833973,   6.2242948 ,
        11.07035322,  43.54441085,  41.39951468,  34.6025522 ])

In [14]:
def state_transitions(machine_states, transition_matrix):
    # Determine new machine states
          for i in range(NUM_MACHINES):
            current_state = machine_states[i]
            if current_state < NUM_DAMAGE_STATES - 1:
                transition_probs = transitions[current_state]
                next_state = np.random.choice([current_state + 1, NUM_DAMAGE_STATES - 1], p=transition_probs)
                machine_states[i] = next_state
            else:
                machine_states[i] = 0  # Reset after breakdown
          return machine_states

**Final Piece of Code**

In [15]:
import numpy as np

# Constants
NUM_MACHINES = 3
NUM_DAMAGE_STATES = 8
NUM_EPISODES = 100
NUM_PROBLEMS = 5120
DISCOUNT_FACTOR = 0.95

In [16]:
def generate_transition_probabilities():
    """
    Generates transition probabilities for a machine.
    Ensures probabilities are ordered decreasingly and sum to 1.
    """
    probs = np.sort(np.random.uniform(0, 1, NUM_DAMAGE_STATES - 1))[::-1]
    return np.column_stack((probs, 1 - probs))  # Two probabilities per state

In [17]:
def initialize_transition_matrices():
    """
    Creates separate transition matrices P0 (no repair) and P1 (with repair) for all machines.
    """
    P0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    P1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        transitions = generate_transition_probabilities()

        # Fill P0 (No repair scenario) for machine m
        for i in range(NUM_DAMAGE_STATES - 1):
            P0_all[m, i, i + 1] = transitions[i, 0]  # Move to worse state
            P0_all[m, i, 0] = transitions[i, 1]  # Reset to pristine

        P0_all[m, -1, 0] = 1  # Most damaged state resets to pristine

        # Fill P1 (With repair scenario) for machine m
        P1_all[m, :, 0] = 1  # Repair resets any state to pristine

    return P0_all, P1_all

In [18]:
P0_all, P1_all = initialize_transition_matrices()

In [19]:
P0_all

array([[[0.06595918, 0.93404082, 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.2738519 , 0.        , 0.7261481 , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.34066294, 0.        , 0.        , 0.65933706, 0.        ,
         0.        , 0.        , 0.        ],
        [0.38065088, 0.        , 0.        , 0.        , 0.61934912,
         0.        , 0.        , 0.        ],
        [0.39321438, 0.        , 0.        , 0.        , 0.        ,
         0.60678562, 0.        , 0.        ],
        [0.56799409, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.43200591, 0.        ],
        [0.65084249, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.34915751],
        [1.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ]],

       [[0.12083511, 0.87916489, 0.        , 0.        , 0.        ,
         

In [20]:
P1_all

array([[[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]],

       [[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]],

       [[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0

In [21]:
def initialize_reward_matrices(P0_all, D, a):
    """
    Initializes the reward matrices R0 (passive) and R1 (repair).
    """
    R0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    R1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        Cm = np.random.uniform(D, D + 25)
        Km = a * Cm

        # Passive Reward Matrix (R0)
        for k in range(NUM_DAMAGE_STATES):
            R0_all[m, k] = P0_all[m, k, 0] * (-Km)  # Penalty when transitioning to pristine

        # Repair Reward Matrix (R1)
        R1_all[m, :] = -Cm  # Repair incurs fixed cost

    return R0_all, R1_all

In [22]:
R0_all, R1_all = initialize_reward_matrices(P0_all, 0, 1.5)

In [23]:
R0_all

array([[ -2.39838533,  -9.9577103 , -12.38707091, -13.84109859,
        -14.29792862, -20.65320937, -23.66571465, -36.36166225],
       [ -1.8974042 ,  -3.31960599,  -7.49967835,  -9.42250114,
         -9.97110405, -10.5017676 , -13.3939106 , -15.70242399],
       [-22.58097756, -23.97842707, -24.94020336, -26.47770282,
        -26.6899209 , -34.15131511, -34.42235354, -36.79304474]])

In [24]:
R1_all

array([[-24.24110817, -24.24110817, -24.24110817, -24.24110817,
        -24.24110817, -24.24110817, -24.24110817, -24.24110817],
       [-10.46828266, -10.46828266, -10.46828266, -10.46828266,
        -10.46828266, -10.46828266, -10.46828266, -10.46828266],
       [-24.52869649, -24.52869649, -24.52869649, -24.52869649,
        -24.52869649, -24.52869649, -24.52869649, -24.52869649]])

In [25]:
def state_transitions(machine_states, P0_all):
    """
    Simulates state transitions for all machines using their transition probabilities.
    """
    new_states = np.zeros(NUM_MACHINES, dtype=int)

    for m in range(NUM_MACHINES):
        current_state = machine_states[m]
        transition_probs = P0_all[m, current_state]  # Use correct machine-specific transition matrix
        new_states[m] = np.random.choice(range(NUM_DAMAGE_STATES), p=transition_probs)

    return new_states

Idleness Criteria : If in the entire episode the repairman is idle for even once, that particular problem instance is taken to be idle.

In [26]:
def simulate(D, a):
    """
    Runs the simulation for a given (D, a) pair and returns the idleness percentage.
    """
    idle_count_episodic = 0  # Track episodes where repairmen are idle

    for _ in range(100):        ################################# CHANGE THIS TO NUM_PROBLEMS
    # Generate unique transition and reward matrices for each machine
      P0_all, P1_all = initialize_transition_matrices()
      R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

      # Compute Whittle indices for each machine
      indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
      for m in range(NUM_MACHINES):
          indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

      # Initialize machine states
      machine_states = np.zeros(NUM_MACHINES, dtype=int)

      # Track if this problem instance had any idle repair step
      problem_was_idle = False

      # Simulate NUM_EPISODES time steps for this problem instance
      for _ in range(NUM_EPISODES):
          machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

          # If all Whittle indices are negative at any time step, mark the problem as idle
          if all(index < 0 for index in machine_indices):
              problem_was_idle = True  # Mark this problem as having at least one idle step

          else:
              # Repair machine with highest index
              repair_target = np.argmax(machine_indices)
              machine_states[repair_target] = 0  # Reset to pristine state

          # Apply state transitions
          machine_states = state_transitions(machine_states, P0_all)

      #  If the repairman was idle at least once in this problem, count it as an idle episode
      if problem_was_idle:
          idle_count_episodic += 1

  # Normalize idleness percentage
    return idle_count_episodic / NUM_PROBLEMS


In [27]:
idleness_percentage = {}

D_values = [0, 10, 20, 25, 30, 40, 50]
a_values = [1.5, 2.0, 3.0, 4.0, 5.0]

for D in D_values:
    for a in a_values:
        print(f"Running simulation for D = {D}, a = {a}")
        idleness_percentage[(D, a)] = simulate(D, a)


Running simulation for D = 0, a = 1.5
Running simulation for D = 0, a = 2.0
Running simulation for D = 0, a = 3.0
Running simulation for D = 0, a = 4.0
Running simulation for D = 0, a = 5.0
Running simulation for D = 10, a = 1.5
Running simulation for D = 10, a = 2.0
Running simulation for D = 10, a = 3.0
Running simulation for D = 10, a = 4.0
Running simulation for D = 10, a = 5.0
Running simulation for D = 20, a = 1.5
Running simulation for D = 20, a = 2.0
Running simulation for D = 20, a = 3.0
Running simulation for D = 20, a = 4.0
Running simulation for D = 20, a = 5.0
Running simulation for D = 25, a = 1.5
Running simulation for D = 25, a = 2.0
Running simulation for D = 25, a = 3.0
Running simulation for D = 25, a = 4.0
Running simulation for D = 25, a = 5.0
Running simulation for D = 30, a = 1.5
Running simulation for D = 30, a = 2.0
Running simulation for D = 30, a = 3.0
Running simulation for D = 30, a = 4.0
Running simulation for D = 30, a = 5.0
Running simulation for D = 40,

In [28]:
idleness_percentage

{(0, 1.5): 0.01953125,
 (0, 2.0): 0.0185546875,
 (0, 3.0): 0.015625,
 (0, 4.0): 0.013671875,
 (0, 5.0): 0.009375,
 (10, 1.5): 0.01953125,
 (10, 2.0): 0.019140625,
 (10, 3.0): 0.0158203125,
 (10, 4.0): 0.0125,
 (10, 5.0): 0.008984375,
 (20, 1.5): 0.01953125,
 (20, 2.0): 0.0189453125,
 (20, 3.0): 0.01484375,
 (20, 4.0): 0.0126953125,
 (20, 5.0): 0.0099609375,
 (25, 1.5): 0.01953125,
 (25, 2.0): 0.0193359375,
 (25, 3.0): 0.016796875,
 (25, 4.0): 0.0138671875,
 (25, 5.0): 0.009375,
 (30, 1.5): 0.01953125,
 (30, 2.0): 0.0177734375,
 (30, 3.0): 0.0154296875,
 (30, 4.0): 0.01171875,
 (30, 5.0): 0.0099609375,
 (40, 1.5): 0.01953125,
 (40, 2.0): 0.0193359375,
 (40, 3.0): 0.015234375,
 (40, 4.0): 0.0134765625,
 (40, 5.0): 0.009375,
 (50, 1.5): 0.01953125,
 (50, 2.0): 0.01875,
 (50, 3.0): 0.0162109375,
 (50, 4.0): 0.0123046875,
 (50, 5.0): 0.008984375}

In [29]:
def simulate_average(D, a):
    """
    Runs the simulation for a given (D, a) pair and returns the idleness percentage.
    """
    idle_count_episodic = 0  # Track episodes where repairmen are idle

    for _ in range(NUM_PROBLEMS):
    # Generate unique transition and reward matrices for each machine
      P0_all, P1_all = initialize_transition_matrices()
      R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

      # Compute Whittle indices for each machine
      indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
      for m in range(NUM_MACHINES):
          indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

      # Initialize machine states
      machine_states = np.zeros(NUM_MACHINES, dtype=int)

      # Track if this problem instance had any idle repair step
      idle_count_per_episode = 0

      # Simulate NUM_EPISODES time steps for this problem instance
      for _ in range(NUM_EPISODES):
          machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

          # If all Whittle indices are negative at any time step, mark the problem as idle
          if all(index < 0 for index in machine_indices):
              idle_count_per_episode += 1   # Mark this problem as having at least one idle step

          else:
              # Repair machine with highest index
              repair_target = np.argmax(machine_indices)
              machine_states[repair_target] = 0  # Reset to pristine state

          # Apply state transitions
          machine_states = state_transitions(machine_states, P0_all)

      #  If the repairman was idle at least once in this problem, count it as an idle episode
      if idle_count_per_episode > 0:
          idle_count_episodic += idle_count_per_episode / NUM_EPISODES

  # Normalize idleness percentage
    return idle_count_episodic / NUM_PROBLEMS


In [30]:
idleness_percentage = {}


for D in D_values:
    for a in a_values:
        print(f"Running simulation for D = {D}, a = {a}")
        idleness_percentage[(D, a)] = simulate_average(D, a)


Running simulation for D = 0, a = 1.5
Running simulation for D = 0, a = 2.0
Running simulation for D = 0, a = 3.0
Running simulation for D = 0, a = 4.0
Running simulation for D = 0, a = 5.0
Running simulation for D = 10, a = 1.5
Running simulation for D = 10, a = 2.0
Running simulation for D = 10, a = 3.0
Running simulation for D = 10, a = 4.0
Running simulation for D = 10, a = 5.0
Running simulation for D = 20, a = 1.5
Running simulation for D = 20, a = 2.0
Running simulation for D = 20, a = 3.0
Running simulation for D = 20, a = 4.0
Running simulation for D = 20, a = 5.0
Running simulation for D = 25, a = 1.5
Running simulation for D = 25, a = 2.0
Running simulation for D = 25, a = 3.0
Running simulation for D = 25, a = 4.0
Running simulation for D = 25, a = 5.0
Running simulation for D = 30, a = 1.5
Running simulation for D = 30, a = 2.0
Running simulation for D = 30, a = 3.0
Running simulation for D = 30, a = 4.0
Running simulation for D = 30, a = 5.0
Running simulation for D = 40,

In [31]:
idleness_percentage

{(0, 1.5): 0.559361328125001,
 (0, 2.0): 0.27485351562499777,
 (0, 3.0): 0.08277148437499934,
 (0, 4.0): 0.03506835937500037,
 (0, 5.0): 0.01846093750000018,
 (10, 1.5): 0.5519042968749991,
 (10, 2.0): 0.27294726562500093,
 (10, 3.0): 0.08447265624999936,
 (10, 4.0): 0.03582812500000036,
 (10, 5.0): 0.018035156250000187,
 (20, 1.5): 0.5562968750000006,
 (20, 2.0): 0.2731054687499993,
 (20, 3.0): 0.08501953124999956,
 (20, 4.0): 0.0344765625000003,
 (20, 5.0): 0.017996093750000143,
 (25, 1.5): 0.5571933593750003,
 (25, 2.0): 0.27496679687499964,
 (25, 3.0): 0.08552343749999934,
 (25, 4.0): 0.03561718750000036,
 (25, 5.0): 0.018970703125000172,
 (30, 1.5): 0.5559746093750004,
 (30, 2.0): 0.27662695312499896,
 (30, 3.0): 0.08450976562499937,
 (30, 4.0): 0.03495507812500033,
 (30, 5.0): 0.01851367187500023,
 (40, 1.5): 0.5539960937499977,
 (40, 2.0): 0.2726777343750001,
 (40, 3.0): 0.0840058593749997,
 (40, 4.0): 0.03541601562500036,
 (40, 5.0): 0.018517578125000225,
 (50, 1.5): 0.55554296

In [32]:
def compute_T_k(p_k):
    """
    Compute expected time to reach state k from the pristine state for a single machine.

    Parameters:
    - p_k: Transition probabilities for a single machine (shape: (NUM_DAMAGE_STATES - 1,))

    Returns:
    - T_k: Expected time to reach state k (shape: (NUM_DAMAGE_STATES - 1,))
    """
    T_k = np.zeros(NUM_DAMAGE_STATES - 1)  # Initialize expected time array

    for k in range(1, NUM_DAMAGE_STATES - 1):  # Iterate over damage states
        for j in range(k):  # Compute time based on previous states
            T_k[k] += p_k[j] * T_k[j]  # Recursive accumulation
        T_k[k] += 1  # Add base step (accounts for at least one step to reach k)

    return T_k

In [33]:
def compute_whittle_indices_paper(p_k, K_m, C_m):
    """
    Computes Whittle indices for a single machine using the formula from Corollary 2.

    Parameters:
    - p_k: Transition probabilities for a single machine (shape: (NUM_DAMAGE_STATES - 1,))
    - K_m: Breakdown cost for the machine
    - C_m: Repair cost for the machine

    Returns:
    - W_m_k: Whittle indices (shape: (NUM_DAMAGE_STATES - 1,))
    """
    T_k = compute_T_k(p_k)  # Compute T(k)
    W_m_k = np.zeros(NUM_DAMAGE_STATES - 1)  # Store Whittle indices

    for k in range(NUM_DAMAGE_STATES - 1):
        term = (T_k[k] - p_k[k] * T_k[k - 1]) if k > 0 else T_k[k]
        W_m_k[k] = K_m * (1 - p_k[k] * term) - C_m

    return W_m_k

In [34]:
def parameters_transition():
    """
    Generates transition matrices P0 (no repair) and P1 (with repair) for all machines.
    Returns:
    - P0_all: Passive transition matrices (shape: NUM_MACHINES x NUM_DAMAGE_STATES x NUM_DAMAGE_STATES)
    - P1_all: Active transition matrices (repair case, resets to pristine state)
    """
    P0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    P1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    p_k_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES - 1))  # Store only the next-state transition probabilities

    for m in range(NUM_MACHINES):
        transitions = generate_transition_probabilities()  # Get unique transition probabilities per machine
        p_k_all[m] = transitions[:, 0]  # Extract probability of transitioning to next state

        for i in range(NUM_DAMAGE_STATES - 1):
            P0_all[m, i, i + 1] = transitions[i, 0]  # Move to worse state
            P0_all[m, i, 0] = transitions[i, 1]  # Jump to pristine state

        P0_all[m, -1, 0] = 1  # Most damaged state resets to pristine
        P1_all[m, :, 0] = 1  # Every state transitions to pristine when repaired

    return P0_all, P1_all, p_k_all


In [35]:
def generate_reward_matrices(P0_all):
    """
    Generates reward matrices R0 (passive) and R1 (active) for all machines.
    Also generates the cost values `C_m` (repair cost) and `K_m` (breakdown cost).

    Parameters:
    - P0_all: Passive transition matrices (NUM_MACHINES x NUM_DAMAGE_STATES x NUM_DAMAGE_STATES)
    - D: Base cost parameter (varies across simulations)

    Returns:
    - R0_all: Passive reward matrices (NUM_MACHINES x NUM_DAMAGE_STATES)
    - R1_all: Active reward matrices (NUM_MACHINES x NUM_DAMAGE_STATES)
    - C_m_values: Repair costs for each machine (NUM_MACHINES,)
    - K_m_values: Breakdown costs for each machine (NUM_MACHINES,)
    """
    R0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    R1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    C_m_values = np.zeros(NUM_MACHINES)
    K_m_values = np.zeros(NUM_MACHINES)

    for m in range(NUM_MACHINES):
        # Generate repair and breakdown costs
        C_m = np.random.uniform(50, 75)  # Repair cost sampled from U(D, D+25)
        K_m = (m + 1) * 1.5 * C_m  # Varying breakdown cost as in the problem description

        C_m_values[m] = C_m
        K_m_values[m] = K_m

        # Compute reward matrices
        for k in range(NUM_DAMAGE_STATES):
            R0_all[m, k] = P0_all[m, k, 0] * (-K_m)  # Passive penalty when resetting to pristine
            R1_all[m, k] = -C_m  # Repair incurs fixed cost

    return R0_all, R1_all, C_m_values, K_m_values


In [36]:
P0_all, P1_all, p_k_all = parameters_transition()
R0_all, R1_all, C_m_values, K_m_values = generate_reward_matrices(P0_all)

In [ ]:
def compute_G_H(p, beta):
    """ Computes G(k) and H(k) based on transition probabilities. """
    num_states = len(p)
    G = np.zeros(num_states + 1)
    H = np.zeros(num_states)

    for k in range(num_states):
        H[k] = beta**(k+1) * np.prod(p[:k])
        G[k+1] = G[k] + (1 - p[k]) * H[k]

    return G, H

In [37]:
def whittle_paper_discounted(K, C, p, beta=0.95):
    """ Computes the Whittle index for a machine. """
    p = np.append(p, 0)
    G, H = compute_G_H(p, beta)
    print("p : ",p)
    print("Length of  p : ", len(p))
    num_states = len(p)
    W = np.zeros(num_states)

    for k in range(num_states):
        #print(f" {k}th Iteration")
        numerator = (1 - G[k+1]) - p[k] * (1 - beta * G[k])
        denominator = (1 - G[k+1]) - beta * p[k] * (1 - G[k])
        W[k] = K * (numerator / denominator) - C

        #print("W[k] : ", W[k])
    #print("Length of W : ", len(W))
    return W

In [38]:
whittle_paper = []
whittle_package = []
for m in range(NUM_MACHINES):
        # Get machine-specific values
        p_k = p_k_all[m]       # Only transition probabilities needed for the paper
        P0, P1 = P0_all[m], P1_all[m]  # Full matrices needed for package
        R0, R1 = R0_all[m], R1_all[m]  # Full reward matrices
        K_m, C_m = K_m_values[m], C_m_values[m]

        # Compute Whittle indices using the paper's formula
        print(f"Machine {m} Whittle Index : {whittle_paper_discounted( K_m, C_m, p_k)}")
        whittle_paper.append(whittle_paper_discounted( K_m, C_m, p_k))

        # Compute Whittle indices using the package
        whittle_package.append(whittle_indices(P0, P1, R0, R1))

NameError: name 'compute_G_H' is not defined

In [ ]:
whittle_paper

[array([-67.89095058, -23.6995825 ,   7.4511254 ,  22.21463077,
         25.73076141,  27.80261993,  28.37622286,  34.86349562]),
 array([-34.51024697,  57.24420616,  73.72492723,  97.03480438,
        113.68595574, 121.11405915, 122.78150597, 124.62289332]),
 array([-56.72463276, -14.17513929,  61.37515406, 141.9912223 ,
        180.60903588, 190.05024321, 204.96216862, 205.75382039])]

In [ ]:
whittle_package

[array([-67.89095058, -23.6995825 ,   7.4511254 ,  22.21463077,
         25.73076141,  27.80261993,  28.37622286,  34.86349562]),
 array([-34.51024697,  57.24420616,  73.72492723,  97.03480438,
        113.68595574, 121.11405915, 122.78150597, 124.62289332]),
 array([-56.72463276, -14.17513929,  61.37515406, 141.9912223 ,
        180.60903588, 190.05024321, 204.96216862, 205.75382039])]

In [ ]:
"""import numpy as np

NUM_MACHINES = 3
NUM_DAMAGE_STATES = 8  # 0 (pristine) to 7 (most damaged)

def state_transitions(machine_states, transition_matrix):
    # Simulate state transitions for all machines using transition probabilities
    new_states = []
    for i in range(NUM_MACHINES):
        current_state = machine_states[i]
        transition_probs = transition_matrix[current_state]  # Use correct transition matrix

        # Determine next state
        next_state = np.random.choice(range(NUM_DAMAGE_STATES), p=transition_probs)

        # 🔹 If we reach the most damaged state, reset to pristine (0)
        if current_state == NUM_DAMAGE_STATES - 1:
            next_state = 0

        new_states.append(next_state)
    return np.array(new_states)

# 🔹 Store idleness percentage as a dictionary
idleness_percentage = {}

for D in D_values:
    for a in a_values:
        print("D : ", D)
        print("Alpha : ", a)

        idle_count_episodic = 0  # Count how many problems experienced idle repairmen

        for p_instance in range(5120):  # Iterate over all problems
            # Generate transition matrices
            P0 = np.zeros((NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
            P1 = np.zeros((NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))

            # Generate transition probabilities
            transitions = generate_transition_probabilities()

            # 🔹 Fill P0 (No repair scenario)
            for i in range(NUM_DAMAGE_STATES - 1):
                P0[i, i + 1] = transitions[i, 0]  # Move to worse state
                P0[i, 0] = transitions[i, 1]  # Jump to pristine (breakdown-like reset)

            # 🔹 Last state (most damaged) resets to pristine
            P0[-1, 0] = 1

            # 🔹 Fill P1 (With repair scenario) - Always goes back to pristine state
            for i in range(NUM_DAMAGE_STATES):
                P1[i, 0] = 1  # Repair moves any state to pristine

            # Cost calculations
            Cm = np.random.uniform(D, D + 25)
            Km = a * Cm

            # 🔹 Corrected Passive Reward Matrix (R0)
            R0 = np.zeros(NUM_DAMAGE_STATES)
            for k in range(NUM_DAMAGE_STATES):
                R0[k] = P0[k, 0] * (-Km)  # 🔹 Penalty applied when transitioning to pristine

            R1 = np.full(NUM_DAMAGE_STATES, -Cm)  # Repair cost is always -Cm

            # Compute Whittle indices
            indices = whittle_indices(P0, P1, R0, R1)

            # Initialize machine states
            machine_states = np.zeros(NUM_MACHINES, dtype=int)

            # Track whether this problem had an idle moment
            problem_was_idle = False

            for _ in range(100):
                machine_indices = [indices[state] for state in machine_states]

                if all(index < 0 for index in machine_indices):
                    problem_was_idle = True  # Mark this problem as having an idle step
                    break  # No need to continue checking further steps for this problem

                else:
                    repair_target = np.argmax(machine_indices)  # Highest index gets repair
                    machine_states[repair_target] = 0  # Reset to pristine state

                # Apply state transitions
                machine_states = state_transitions(machine_states, P0)

            if problem_was_idle:
                idle_count_episodic += 1  # Count this problem as idle

        # 🔹 Store result in dictionary with (D, a) as key
        idleness_percentage[(D, a)] = idle_count_episodic / 5120  # Normalize to fraction

# 🔹 Print the dictionary for reference
print("\nIdleness Percentage Dictionary:")
for key, value in idleness_percentage.items():
    print(f"D={key[0]}, Alpha={key[1]} → Idleness %: {value:.4f}")
"""

D :  0
Alpha :  1.5
D :  0
Alpha :  2.0
D :  0
Alpha :  3.0
D :  0
Alpha :  4.0
D :  0
Alpha :  5.0
D :  10
Alpha :  1.5
D :  10
Alpha :  2.0
D :  10
Alpha :  3.0
D :  10
Alpha :  4.0
D :  10
Alpha :  5.0
D :  20
Alpha :  1.5
D :  20
Alpha :  2.0
D :  20
Alpha :  3.0
D :  20
Alpha :  4.0
D :  20
Alpha :  5.0
D :  25
Alpha :  1.5
D :  25
Alpha :  2.0
D :  25
Alpha :  3.0
D :  25
Alpha :  4.0
D :  25
Alpha :  5.0
D :  30
Alpha :  1.5
D :  30
Alpha :  2.0
D :  30
Alpha :  3.0
D :  30
Alpha :  4.0
D :  30
Alpha :  5.0
D :  40
Alpha :  1.5
D :  40
Alpha :  2.0
D :  40
Alpha :  3.0
D :  40
Alpha :  4.0
D :  40
Alpha :  5.0
D :  50
Alpha :  1.5
D :  50
Alpha :  2.0
D :  50
Alpha :  3.0
D :  50
Alpha :  4.0
D :  50
Alpha :  5.0

Idleness Percentage Dictionary:
D=0, Alpha=1.5 → Idleness %: 0.9998
D=0, Alpha=2.0 → Idleness %: 0.9887
D=0, Alpha=3.0 → Idleness %: 0.9385
D=0, Alpha=4.0 → Idleness %: 0.8676
D=0, Alpha=5.0 → Idleness %: 0.7934
D=10, Alpha=1.5 → Idleness %: 0.9998
D=10, Alpha=2.0 → Id

**Numerical Study**

In [ ]:
NUM_MACHINES = 3
NUM_DAMAGE_STATES = 8  # Assuming 3 possible states per machine
NUM_EPISODES = 100  # Time steps for each simulation
NUM_PROBLEMS = 512  # Number of state combinations (8^3)
NUM_SIMULATIONS = 100  # Number of transition matrices to simulate

In [ ]:
def generate_machine_states():
    """
    Generates all possible machine states combinations for NUM_MACHINES machines.
    Each machine has NUM_DAMAGE_STATES possible states, resulting in NUM_PROBLEMS combinations.
    """
    grid = np.meshgrid(*[range(NUM_DAMAGE_STATES)] * NUM_MACHINES)

    # Stack the grid arrays together and reshape to get all combinations as rows
    states = np.vstack([g.flatten() for g in grid]).T
    return states

In [ ]:
states = generate_machine_states()

In [ ]:
states

array([[0, 0, 0],
       [0, 0, 1],
       [0, 0, 2],
       ...,
       [7, 7, 5],
       [7, 7, 6],
       [7, 7, 7]])

In [ ]:
len(states)

27

In [ ]:
def simulate_single_repairman(D, a):

    idleness_count = 0
    all_machine_states = generate_machine_states()

    print(f"Number of states : {len(all_machine_states)}")

    for state in all_machine_states:
        machine_states = state.copy()

        print(f"Running simulation for states : {machine_states}")

        for simulation in range(NUM_SIMULATIONS):

            P0_all, P1_all = initialize_transition_matrices()
            R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)
            problem_was_idle = False

            # Compute Whittle indices for each machine
            indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

            for m in range(NUM_MACHINES):
                indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

            machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

            if all(index < 0 for index in machine_indices):
                problem_was_idle = True

            if problem_was_idle:
              idleness_count += 1

    return idleness_count /  (NUM_SIMULATIONS * len(all_machine_states))

In [ ]:
idleness_percentage = {}

D_values = [0, 10, 20, 25, 30, 40, 50]
a_values = [1.5, 2.0, 3.0, 4.0, 5.0]

# Running the simulation for each (D, a) pair
for D in D_values:
    for a in a_values:
        print(f"Running simulation for D = {D}, a = {a}")
        idleness_percentage[(D, a)] = simulate(D, a)

Streaming output truncated to the last 5000 lines.
Running simulation for states : [1 2 2]
Running simulation for states : [1 2 3]
Running simulation for states : [1 2 4]
Running simulation for states : [1 2 5]
Running simulation for states : [1 2 6]
Running simulation for states : [1 2 7]
Running simulation for states : [2 2 0]
Running simulation for states : [2 2 1]
Running simulation for states : [2 2 2]
Running simulation for states : [2 2 3]
Running simulation for states : [2 2 4]
Running simulation for states : [2 2 5]
Running simulation for states : [2 2 6]
Running simulation for states : [2 2 7]
Running simulation for states : [3 2 0]
Running simulation for states : [3 2 1]
Running simulation for states : [3 2 2]
Running simulation for states : [3 2 3]
Running simulation for states : [3 2 4]
Running simulation for states : [3 2 5]
Running simulation for states : [3 2 6]
Running simulation for states : [3 2 7]
Running simulation for states : [4 2 0]
Running simulation for states

In [ ]:
idleness_percentage

{(0, 1.5): 0.071484375,
 (0, 2.0): 0.0271484375,
 (0, 3.0): 0.009765625,
 (0, 4.0): 0.0044921875,
 (0, 5.0): 0.0021484375,
 (10, 1.5): 0.0796875,
 (10, 2.0): 0.0283203125,
 (10, 3.0): 0.008984375,
 (10, 4.0): 0.00546875,
 (10, 5.0): 0.001953125,
 (20, 1.5): 0.0740234375,
 (20, 2.0): 0.027734375,
 (20, 3.0): 0.00859375,
 (20, 4.0): 0.0046875,
 (20, 5.0): 0.0033203125,
 (25, 1.5): 0.073828125,
 (25, 2.0): 0.0302734375,
 (25, 3.0): 0.010546875,
 (25, 4.0): 0.0029296875,
 (25, 5.0): 0.00234375,
 (30, 1.5): 0.0802734375,
 (30, 2.0): 0.0294921875,
 (30, 3.0): 0.0123046875,
 (30, 4.0): 0.00390625,
 (30, 5.0): 0.001953125,
 (40, 1.5): 0.076171875,
 (40, 2.0): 0.0296875,
 (40, 3.0): 0.0095703125,
 (40, 4.0): 0.005078125,
 (40, 5.0): 0.0033203125,
 (50, 1.5): 0.0806640625,
 (50, 2.0): 0.0296875,
 (50, 3.0): 0.0103515625,
 (50, 4.0): 0.00390625,
 (50, 5.0): 0.00234375}

In [ ]:
for keys, values in idleness_percentage.items():
    idleness_percentage[keys] = values * 100

In [ ]:
idleness_percentage

{(0, 1.5): 7.1484375,
 (0, 2.0): 2.71484375,
 (0, 3.0): 0.9765625,
 (0, 4.0): 0.44921874999999994,
 (0, 5.0): 0.21484375000000003,
 (10, 1.5): 7.968749999999999,
 (10, 2.0): 2.83203125,
 (10, 3.0): 0.8984374999999999,
 (10, 4.0): 0.546875,
 (10, 5.0): 0.1953125,
 (20, 1.5): 7.40234375,
 (20, 2.0): 2.7734375,
 (20, 3.0): 0.8593750000000001,
 (20, 4.0): 0.46875,
 (20, 5.0): 0.33203125,
 (25, 1.5): 7.382812499999999,
 (25, 2.0): 3.02734375,
 (25, 3.0): 1.0546875,
 (25, 4.0): 0.29296875,
 (25, 5.0): 0.234375,
 (30, 1.5): 8.02734375,
 (30, 2.0): 2.94921875,
 (30, 3.0): 1.23046875,
 (30, 4.0): 0.390625,
 (30, 5.0): 0.1953125,
 (40, 1.5): 7.6171875,
 (40, 2.0): 2.96875,
 (40, 3.0): 0.9570312500000001,
 (40, 4.0): 0.5078125,
 (40, 5.0): 0.33203125,
 (50, 1.5): 8.06640625,
 (50, 2.0): 2.96875,
 (50, 3.0): 1.03515625,
 (50, 4.0): 0.390625,
 (50, 5.0): 0.234375}

In [ ]:
NUM_MACHINES = 4
NUM_DAMAGE_STATES = 8
NUM_SIMULATIONS = 100
DISCOUNT_FACTOR = 0.95

In [ ]:

def simulate_multiple_repairmen(D, a):

    single_idle_count = 0
    both_idle_count = 0

    all_machine_states = generate_machine_states()

    print(f"Number of states : {len(all_machine_states)}")

    for state in all_machine_states:
        machine_states = state.copy()

        print(f"Running simulation for states : {machine_states}")

        for simulation in range(NUM_SIMULATIONS):

            P0_all, P1_all = initialize_transition_matrices()
            R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

            single_idle = False
            both_idle = False

            # Compute Whittle indices for each machine
            indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

            for m in range(NUM_MACHINES):
                indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

            machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

            sorted_indices = sorted(machine_indices, reverse=True)

            if sorted_indices[1] < 0:  # If the second repairman’s best option is still negative
                if sorted_indices[0] < 0:  # If even the first repairman has a negative index
                    both_idle_count += 1  # Both repairmen are idle
                else:
                    single_idle_count += 1  # Only one repairman is idle

    total_simulations = NUM_SIMULATIONS * len(all_machine_states)

    return both_idle_count / total_simulations, single_idle_count / total_simulations

In [ ]:
D_values = [0, 10, 20, 25, 30, 40, 50]
a_values = [1.5, 2.0, 3.0, 4.0, 5.0]

In [ ]:
single_idleness_percentage = {}
both_idleness_percentage = {}

for d in D_values:
    for a in a_values:
        print(f"Running simulation for D = {d}, a = {a}")
        both_idle_count, single_idle_count = simulate_multiple_repairmen(d, a)
        both_idleness_percentage[(d, a)] = both_idle_count
        single_idleness_percentage[(d, a)] = single_idle_count

Streaming output truncated to the last 5000 lines.
Running simulation for states : [1 6 7 2]
Running simulation for states : [1 6 7 3]
Running simulation for states : [1 6 7 4]
Running simulation for states : [1 6 7 5]
Running simulation for states : [1 6 7 6]
Running simulation for states : [1 6 7 7]
Running simulation for states : [2 6 0 0]
Running simulation for states : [2 6 0 1]
Running simulation for states : [2 6 0 2]
Running simulation for states : [2 6 0 3]
Running simulation for states : [2 6 0 4]
Running simulation for states : [2 6 0 5]
Running simulation for states : [2 6 0 6]
Running simulation for states : [2 6 0 7]
Running simulation for states : [2 6 1 0]
Running simulation for states : [2 6 1 1]
Running simulation for states : [2 6 1 2]
Running simulation for states : [2 6 1 3]
Running simulation for states : [2 6 1 4]
Running simulation for states : [2 6 1 5]
Running simulation for states : [2 6 1 6]
Running simulation for states : [2 6 1 7]
Running simulation for st

In [ ]:
single_idleness_percentage

{(0, 1.5): 0.1746826171875,
 (0, 2.0): 0.0826416015625,
 (0, 3.0): 0.031689453125,
 (0, 4.0): 0.016064453125,
 (0, 5.0): 0.0094970703125,
 (10, 1.5): 0.17373046875,
 (10, 2.0): 0.082373046875,
 (10, 3.0): 0.0318603515625,
 (10, 4.0): 0.0155517578125,
 (10, 5.0): 0.00908203125,
 (20, 1.5): 0.1739990234375,
 (20, 2.0): 0.0795654296875,
 (20, 3.0): 0.0318115234375,
 (20, 4.0): 0.0151611328125,
 (20, 5.0): 0.0091552734375,
 (25, 1.5): 0.17255859375,
 (25, 2.0): 0.080859375,
 (25, 3.0): 0.0322021484375,
 (25, 4.0): 0.0149658203125,
 (25, 5.0): 0.01044921875,
 (30, 1.5): 0.17314453125,
 (30, 2.0): 0.080810546875,
 (30, 3.0): 0.0302978515625,
 (30, 4.0): 0.0171875,
 (30, 5.0): 0.009033203125,
 (40, 1.5): 0.17451171875,
 (40, 2.0): 0.0799072265625,
 (40, 3.0): 0.03125,
 (40, 4.0): 0.016064453125,
 (40, 5.0): 0.0090576171875,
 (50, 1.5): 0.176318359375,
 (50, 2.0): 0.080810546875,
 (50, 3.0): 0.0296142578125,
 (50, 4.0): 0.015625,
 (50, 5.0): 0.0089599609375}

In [ ]:
both_idleness_percentage

{(0, 1.5): 0.0310302734375,
 (0, 2.0): 0.00869140625,
 (0, 3.0): 0.001953125,
 (0, 4.0): 0.0005859375,
 (0, 5.0): 0.0003173828125,
 (10, 1.5): 0.0314208984375,
 (10, 2.0): 0.009375,
 (10, 3.0): 0.0024169921875,
 (10, 4.0): 0.0007568359375,
 (10, 5.0): 0.000537109375,
 (20, 1.5): 0.0319091796875,
 (20, 2.0): 0.009228515625,
 (20, 3.0): 0.002099609375,
 (20, 4.0): 0.0008056640625,
 (20, 5.0): 0.00029296875,
 (25, 1.5): 0.0321533203125,
 (25, 2.0): 0.0097900390625,
 (25, 3.0): 0.002001953125,
 (25, 4.0): 0.0008544921875,
 (25, 5.0): 0.0002197265625,
 (30, 1.5): 0.0326171875,
 (30, 2.0): 0.00888671875,
 (30, 3.0): 0.0023681640625,
 (30, 4.0): 0.000830078125,
 (30, 5.0): 0.0003662109375,
 (40, 1.5): 0.03125,
 (40, 2.0): 0.0097412109375,
 (40, 3.0): 0.0021484375,
 (40, 4.0): 0.000732421875,
 (40, 5.0): 0.0002685546875,
 (50, 1.5): 0.0320068359375,
 (50, 2.0): 0.0089111328125,
 (50, 3.0): 0.0022216796875,
 (50, 4.0): 0.0007080078125,
 (50, 5.0): 0.000537109375}

In [ ]:
def calculate_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all):
    cost = 0
    all_machine_states = generate_machine_states()

    for state in all_machine_states:
        machine_states = state.copy()

        for simulation in range(NUM_SIMULATIONS):
            # Compute Whittle indices for each machine
            indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

            for m in range(NUM_MACHINES):
                indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

            # Compute the Whittle indices for the current state of each machine
            machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

            # Find the machine with the highest Whittle index
            highest_index_machine = np.argmax(machine_indices)

            # Compute total cost
            if all(index < 0 for index in machine_indices):  # Idle case
                for m in range(NUM_MACHINES):
                    cost += R0_all[m, machine_states[m]]
            else:
                for m in range(NUM_MACHINES):
                    cost += R0_all[m, machine_states[m]]

                # Active cost for the selected machine
                cost += (R1_all[highest_index_machine, machine_states[highest_index_machine]]
                         - R0_all[highest_index_machine, machine_states[highest_index_machine]])

    return cost


In [ ]:
def compute_optimal_cost(D, a, P0_all, P1_all, R0_all, R1_all, max_iterations=1000, gamma=0.95, tol=1e-6):
    all_machine_states = generate_machine_states()
    num_states = len(all_machine_states)  # Number of states
    V = np.zeros(num_states)  # Initialize value function

    for iteration in range(max_iterations):
        V_old = V.copy()
        max_diff = 0

        for s, state in enumerate(all_machine_states):
            m1, m2, m3 = state  # Extract machine states

            cost_idle = (
                R0_all[0, m1] + R0_all[1, m2] + R0_all[2, m3]
                + gamma * (
                    np.sum(P0_all[0, m1, :] * V_old) +
                    np.sum(P0_all[1, m2, :] * V_old) +
                    np.sum(P0_all[2, m3, :] * V_old)
                )
            )

            cost_active = (
                R1_all[0, m1] + R1_all[1, m2] + R1_all[2, m3]
                + gamma * (
                    np.sum(P1_all[0, m1, :] * V_old) +
                    np.sum(P1_all[1, m2, :] * V_old) +
                    np.sum(P1_all[2, m3, :] * V_old)
                )
            )

            # Bellman update
            V[s] = min(cost_idle, cost_active)

            max_diff = max(max_diff, abs(V[s] - V_old[s]))

        # Clip every 10 iterations to avoid slow convergence
        if iteration % 10 == 0:
            V = np.clip(V, -1e6, 1e6)

        print(f"Iteration {iteration}: max(V)={np.max(V)}, min(V)={np.min(V)}")

        if max_diff < tol:
            print(f"Converged after {iteration + 1} iterations.")
            break

    # Final clipping after convergence
    V = np.clip(V, -1e6, 1e6)
    return V

In [ ]:
def cost_comparison(D, a):
    # Initialize the same transition probabilities and rewards for both computations
    P0_all, P1_all = initialize_transition_matrices()
    R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

    # Compute costs for both policies
    whittle_cost = calculate_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all)
    optimal_cost = compute_optimal_cost(D, a, P0_all, P1_all, R0_all, R1_all)

    return whittle_cost, optimal_cost

In [ ]:
whittle_cost = {}
optimal_cost = {}

for d in D_values:
    for a in a_values:
        print(f"Running simulation for D = {d}, a = {a}")
        whittle_cost[(d, a)], optimal_cost[(d, a)] = cost_comparison(d, a)

Running simulation for D = 0, a = 1.5


ValueError: operands could not be broadcast together with shapes (8,) (512,) 

In [ ]:
whittle_cost

{(0, 1.5): np.float64(-70758.11817150582),
 (0, 2.0): np.float64(-39172.455622032394),
 (0, 3.0): np.float64(-59066.048834876994),
 (0, 4.0): np.float64(-351165.0231779717),
 (0, 5.0): np.float64(-245717.45432377982),
 (10, 1.5): np.float64(-188218.14831936933),
 (10, 2.0): np.float64(-299125.73513948417),
 (10, 3.0): np.float64(-290612.8577440902),
 (10, 4.0): np.float64(-415430.5560353594),
 (10, 5.0): np.float64(-677323.1054920058),
 (20, 1.5): np.float64(-318948.3635145708),
 (20, 2.0): np.float64(-517710.92583617184),
 (20, 3.0): np.float64(-454611.4593524096),
 (20, 4.0): np.float64(-640530.657938449),
 (20, 5.0): np.float64(-1174454.3203073854),
 (25, 1.5): np.float64(-473225.1279671671),
 (25, 2.0): np.float64(-537740.4103409357),
 (25, 3.0): np.float64(-681772.0001162335),
 (25, 4.0): np.float64(-749664.1401186119),
 (25, 5.0): np.float64(-840501.8911526732),
 (30, 1.5): np.float64(-492220.3097681367),
 (30, 2.0): np.float64(-596873.7086146821),
 (30, 3.0): np.float64(-632859.

In [ ]:
optimal_cost

{(0,
  1.5): array([-1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -1000000., -1000000., -1000000., -1000000., -1000000.,
        -1000000., -100000

In [ ]:
suboptimal_gaps = {key: whittle_cost[key] - optimal_cost[key] for key in optimal_cost}

# Compute quartiles
gap_values = list(suboptimal_gaps.values())

q1 = np.percentile(gap_values, 25)  # Lower quartile
q2 = np.percentile(gap_values, 50)  # Median
q3 = np.percentile(gap_values, 75)  # Upper quartile

# Print results
print(f"Suboptimal Gaps: {suboptimal_gaps}")
print(f"Lower Quartile (Q1): {q1}")
print(f"Median (Q2): {q2}")
print(f"Upper Quartile (Q3): {q3}")


Suboptimal Gaps: {(0, 1.5): array([929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.88182849,
       929241.88182849, 929241.88182849, 929241.88182849, 929241.8818284

In [ ]:
import numpy as np

NUM_MACHINES = 3
NUM_DAMAGE_STATES = 8
NUM_PROBLEM_INSTANCES = 10  # 10 problem instances per (D, a) pair

In [ ]:
def generate_machine_states():
    """
    Generates all possible machine states combinations for NUM_MACHINES machines.
    Each machine has NUM_DAMAGE_STATES possible states, resulting in NUM_PROBLEMS combinations.
    """
    grid = np.meshgrid(*[range(NUM_DAMAGE_STATES)] * NUM_MACHINES)

    # Stack the grid arrays together and reshape to get all combinations as rows
    states = np.vstack([g.flatten() for g in grid]).T
    return states

In [ ]:
def compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all, max_iterations=10, gamma=0.95, tol=1e-6):
    """
    This code will output a vector of size 512
    """
    """Computes the optimal cost using Value Iteration."""

    all_machine_states = generate_machine_states()
    #print(f"all_machine_states.shape : {all_machine_states.shape}")

    state_to_index = {tuple(state): idx for idx, state in enumerate(all_machine_states)}
    num_states = len(all_machine_states)
    V = np.zeros(num_states) # V opt
    V_whittle = np.zeros(num_states) ## ADDITION 1
    indices_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

    """
    Initialize whittle indices and define 'whittle_action_profile'
    """
    for m in range(NUM_MACHINES):
            indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

    print(f"Indices all : {indices_all}")

    for iteration in range(max_iterations):
        V_old = V.copy()
        V_whittle_old = V_whittle.copy()
        max_diff = 0

        if iteration % 10 == 0:
          print(f"Iteration {iteration}: max(V)={np.max(V)}, min(V)={np.min(V)}")

        for s, state in enumerate(all_machine_states):
            machine_states = state.copy()

            cost_of_all_actions = []

            #print(f"Machine states : {[all_machine_states[m] for m in state]}")

            machine_indices = [indices_all[m, state] for m, state in enumerate(machine_states)]

            if all(index < 0 for index in machine_indices):
                whittle_action_profile = [0, 0, 0]
            else:
                max_index = np.argmax(machine_indices)

                whittle_action_profile = [1 if m == max_index else 0 for m in state]

            # Compute cost when active
            for activation_profile in [[0,0,0], [1,0,0], [0,1,0], [0,0,1]]:
              """compute the cost of activating the arms in "activation profile"""
              R = [R0_all[arm, state[arm]] if activation_profile[arm] == 0 else R1_all[arm, state[arm]] for arm in [0,1,2]]
              P = [P0_all[arm, state[arm]] if activation_profile[arm] == 0 else P1_all[arm, state[arm]] for arm in [0,1,2]]

              #print(f"P.shape: {np.asarray(P).shape}")
              #print(f"R.shape: {np.asarray(R).shape}")

              value_sum_active = 0
              value_whittle = 0

              for next_s1 in range(NUM_DAMAGE_STATES):
                  for next_s2 in range(NUM_DAMAGE_STATES):
                      for next_s3 in range(NUM_DAMAGE_STATES):
                          next_state_index = state_to_index[(next_s1, next_s2, next_s3)]
                          #print(f"Value_sum_Active = {P[0][next_s1] * P[1][next_s2] * P[2][next_s3] * V_old[next_state_index]}")
                          value_sum_active += (
                              P[0][next_s1] * P[1][next_s2] * P[2][next_s3] * V_old[next_state_index]
                          )
                          value_whittle += P[0][next_s1] * P[1][next_s2] * P[2][next_s3] * V_whittle_old[next_state_index]

              cost_of_all_actions.append(sum(R) + gamma * value_sum_active)

              if activation_profile == whittle_action_profile:
                V_whittle[s] = sum(R) + gamma * value_whittle

            # Bellman update
            V[s] = min(cost_of_all_actions)
            print("1111111111111111111")
            max_diff = np.max(max_diff, abs(V[s] - V_old[s])) + np.max(abs(V_whittle - V_whittle_old))

        if max_diff < tol:
            print(f"Converged after {iteration + 1} iterations.")
            break

    return V, V_whittle  # Return mean optimal cost over all states

In [ ]:
def analyze_suboptimality(D, a):
    """Computes the percentage cost suboptimality for 10 problem instances and calculates interquartile ranges."""
    suboptimalities = []
    idle_percentages = []
    print(f"D = {D}, a = {a}")
    for _ in range(NUM_PROBLEM_INSTANCES):

        P0_all, P1_all = initialize_transition_matrices()
        R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

        V_optimal, whittle_cost = compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all)
        print(f"V_optimal: {V_optimal.mean()}, whittle_cost: {whittle_cost.mean()}")


        suboptimality = 100 * (whittle_cost - V_optimal) / V_optimal
        suboptimalities.append(suboptimality)
        print(f"Suboptimalities.len: {len(suboptimalities)}")  # Print first 10 values

    # Compute interquartile statistics
    lower_quartile = np.percentile(suboptimalities, 25)
    median = np.percentile(suboptimalities, 50)
    upper_quartile = np.percentile(suboptimalities, 75)


    print(f"(Lower Quartile: {lower_quartile:.2f}, Median: {median:.2f}, Upper Quartile: {upper_quartile:.2f})")

    return lower_quartile, median, upper_quartile


In [ ]:
D_v = [0, 10]
a_v = [1.5, 2.0]
l_q = {}
m = {}
u_q = {}
i_p = {}
"""
for D in D_v:
  for a in a_v:
    print(f"Running simulation for D = {D}, a = {a}")
"""

lower_q, med, upper_q = analyze_suboptimality(0, 1.5)
l_q[(D, a)] = lower_q
m[(D, a)] = med
u_q[(D, a)] = upper_q

D = 0, a = 1.5
Indices all : [[ -9.59150219  -5.80993668  -2.31182283  -0.83959829   4.33007909
    8.13029489   8.41144778  10.92740492]
 [-12.81466509 -12.68406382  -8.56491835   1.47168478   9.1350465
    9.29984558  10.45690667  11.27044172]
 [-10.78528776  -0.69052184   2.27907076   2.47940453   4.34475262
    5.68924349   6.00706067   6.03253451]]
Iteration 0: max(V)=0.0, min(V)=0.0
1111111111111111111


TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [ ]:
lower_q

np.float64(-56.35910866402774)

In [ ]:
upper_q

np.float64(-42.839815011448266)

In [ ]:
med

np.float64(-48.390874951865584)

In [ ]:
len(suboptimal_gaps)

NameError: name 'suboptimal_gaps' is not defined

In [ ]:
import numpy as np

def generate_machine_states(num_damage_states=8, num_machines=3):
    """Generate all possible machine states"""
    from itertools import product
    return np.array(list(product(range(num_damage_states), repeat=num_machines)))

def initialize_transition_matrices():
    """
    Creates separate transition matrices P0 (no repair) and P1 (with repair) for all machines.
    """
    P0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    P1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        transitions = generate_transition_probabilities()

        # Fill P0 (No repair scenario) for machine m
        for i in range(NUM_DAMAGE_STATES - 1):
            P0_all[m, i, i + 1] = transitions[i, 0]  # Move to worse state
            P0_all[m, i, 0] = transitions[i, 1]  # Reset to pristine

        P0_all[m, -1, 0] = 1  # Most damaged state resets to pristine

        # Fill P1 (With repair scenario) for machine m
        P1_all[m, :, 0] = 1  # Repair resets any state to pristine

    return P0_all, P1_all

def initialize_reward_matrices(P0_all, D, a):
    """
    Initializes the reward matrices R0 (passive) and R1 (repair).
    """
    R0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    R1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        Cm = np.random.uniform(D, D + 25)
        Km = a * Cm

        # Passive Reward Matrix (R0)
        for k in range(NUM_DAMAGE_STATES):
            R0_all[m, k] = P0_all[m, k, 0] * (-Km)  # Penalty when transitioning to pristine

        # Repair Reward Matrix (R1)
        R1_all[m, :] = -Cm  # Repair incurs fixed cost

    return R0_all, R1_all

def whittle_indices(P0, P1, R0, R1):
  model = bandit.restless_bandit_from_P0P1_R0R1(P0, P1, R0, R1)
  whittle_indices = model.whittle_indices(discount = 0.95)
  return whittle_indices

def compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all, max_iterations=1000, gamma=0.95, tol=1e-6):
    """Compute optimal and Whittle index costs using Value Iteration"""
    all_machine_states = generate_machine_states()
    state_to_index = {tuple(state): idx for idx, state in enumerate(all_machine_states)}
    num_states = len(all_machine_states)

    V = np.zeros(num_states)
    V_whittle = np.zeros(num_states)
    indices_all = np.zeros((P0_all.shape[0], P0_all.shape[1]))

    for m in range(P0_all.shape[0]):
        indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

    for iteration in range(max_iterations):
        V_old = V.copy()
        V_whittle_old = V_whittle.copy()
        max_diff = 0

        for s, state in enumerate(all_machine_states):
            machine_indices = [indices_all[m, state[m]] for m in range(len(state))]
            whittle_action = [1 if max(machine_indices) == i else 0 for i in machine_indices]

            cost_values = []
            for action in [[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]:
                R = [R0_all[m, state[m]] if action[m] == 0 else R1_all[m, state[m]] for m in range(len(state))]
                #P = [P0_all[m, state[m]] if action[m] == 0 else P1_all[m, state[m]] for m in range(len(state))]
                P = [P0_all[arm, state[arm]] if action[arm] == 0 else P1_all[arm, state[arm]] for arm in [0,1,2]]

                expected_value = sum(P[0][ns1] * P[1][ns2] * P[2][ns3] * V_old[state_to_index[(ns1, ns2, ns3)]]
                                     for ns1 in range(len(P0_all[0])) for ns2 in range(len(P0_all[1])) for ns3 in range(len(P0_all[2])))

                cost_values.append(sum(R) + gamma * expected_value)
                cost_values = list(np.clip(cost_values, -1e6, 1e6))

                if action == whittle_action:
                    V_whittle[s] = sum(R) + gamma * expected_value

            V[s] = min(cost_values)
            max_diff = max(max_diff, abs(V[s] - V_old[s]) + np.max(abs(V_whittle - V_whittle_old)))

        if max_diff < tol:
            break

    return V, V_whittle

def analyze_suboptimality(D, a, num_problems=10):
    """Compute the percentage cost suboptimality for multiple problem instances"""
    suboptimalities = []

    for _ in range(num_problems):
        P0_all, P1_all = initialize_transition_matrices()
        R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)

        V_optimal, V_whittle = compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all)

        print(f"V_whittle.shape = {V_whittle.shape}, V_optimal.shape = {V_optimal.shape}")
        print(f"V_optimal: {V_optimal.mean()}, V_whittle: {V_whittle.mean()}")

        suboptimalities.append(100 * (V_whittle - V_optimal) / V_optimal)

    lower_quartile = np.percentile(suboptimalities, 25)
    median = np.percentile(suboptimalities, 50)
    upper_quartile = np.percentile(suboptimalities, 75)

    print(f"Suboptimality (Lower Quartile: {lower_quartile:.2f}, Median: {median:.2f}, Upper Quartile: {upper_quartile:.2f})")
    return lower_quartile, median, upper_quartile

# Example Run
D, a = 10, 2.0
analyze_suboptimality(D, a)


Suboptimality (Lower Quartile: -2.67, Median: -2.19, Upper Quartile: -1.80)


(np.float64(-2.6698217335091146),
 np.float64(-2.192339684628771),
 np.float64(-1.803404832538348))

In [ ]:
import numpy as np

In [ ]:
def generate_machine_states(num_damage_states=8, num_machines=3):
    """Generate all possible machine states"""
    from itertools import product
    return np.array(list(product(range(num_damage_states), repeat=num_machines)))

In [ ]:
machine_states = generate_machine_states()

In [ ]:
machine_states

array([[0, 0, 0],
       [0, 0, 1],
       [0, 0, 2],
       ...,
       [7, 7, 5],
       [7, 7, 6],
       [7, 7, 7]])

In [ ]:
len(machine_states)

512

In [ ]:
def initialize_transition_matrices():
    """
    Creates separate transition matrices P0 (no repair) and P1 (with repair) for all machines.
    """
    P0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    P1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        transitions = generate_transition_probabilities()

        # Fill P0 (No repair scenario) for machine m
        for i in range(NUM_DAMAGE_STATES - 1):
            P0_all[m, i, i + 1] = transitions[i, 0]  # Move to worse state
            P0_all[m, i, 0] = transitions[i, 1]  # Reset to pristine

        P0_all[m, -1, 0] = 1  # Most damaged state resets to pristine

        # Fill P1 (With repair scenario) for machine m
        P1_all[m, :, 0] = 1  # Repair resets any state to pristine

    return P0_all, P1_all

In [ ]:
P0_all, P1_all = initialize_transition_matrices()

In [ ]:
P0_all

array([[[0.03878666, 0.96121334, 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.05539871, 0.        , 0.94460129, 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.27045945, 0.        , 0.        , 0.72954055, 0.        ,
         0.        , 0.        , 0.        ],
        [0.6658216 , 0.        , 0.        , 0.        , 0.3341784 ,
         0.        , 0.        , 0.        ],
        [0.68219891, 0.        , 0.        , 0.        , 0.        ,
         0.31780109, 0.        , 0.        ],
        [0.84441993, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.15558007, 0.        ],
        [0.86100904, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.13899096],
        [1.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ]],

       [[0.09788431, 0.90211569, 0.        , 0.        , 0.        ,
         

In [ ]:
P0_all.shape

(3, 8, 8)

In [ ]:
P1_all

array([[[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]],

       [[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.]],

       [[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0

In [ ]:
def initialize_reward_matrices(P0_all, D, a):
    """
    Initializes the reward matrices R0 (passive) and R1 (repair).
    """
    R0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    R1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        Cm = np.random.uniform(D, D + 25)
        Km = a * Cm

        # Passive Reward Matrix (R0)
        for k in range(NUM_DAMAGE_STATES):
            R0_all[m, k] = P0_all[m, k, 0] * (-Km)  # Penalty when transitioning to pristine

        # Repair Reward Matrix (R1)
        R1_all[m, :] = -Cm  # Repair incurs fixed cost

    return R0_all, R1_all

In [ ]:
R0_all, R1_all = initialize_reward_matrices(P0_all, 0, 1.5)

In [ ]:
R0_all

array([[ -0.25569339,  -0.36520506,  -1.78295054,  -4.3892974 ,
         -4.4972616 ,  -5.56667167,  -5.67603209,  -6.59230253],
       [ -3.32315362,  -6.49130135,  -7.04947433,  -8.9642951 ,
        -15.48593953, -22.25851689, -28.72180582, -33.94981009],
       [ -0.53950822,  -2.32764588,  -3.80021711,  -4.5140322 ,
         -5.73522973,  -5.99864222,  -6.17544229,  -8.00797561]])

In [ ]:
R0_all.shape

(3, 8)

In [ ]:
R1_all

array([[ -4.39486836,  -4.39486836,  -4.39486836,  -4.39486836,
         -4.39486836,  -4.39486836,  -4.39486836,  -4.39486836],
       [-22.63320673, -22.63320673, -22.63320673, -22.63320673,
        -22.63320673, -22.63320673, -22.63320673, -22.63320673],
       [ -5.33865041,  -5.33865041,  -5.33865041,  -5.33865041,
         -5.33865041,  -5.33865041,  -5.33865041,  -5.33865041]])

In [ ]:
R1_all.shape

(3, 8)

In [ ]:
import cvxpy as cp
import numpy as np

def solve_restless_bandits_lp(P0_all, P1_all, R0_all, R1_all):
    """
    Solves the restless bandits problem using Linear Programming.

    Parameters:
    - P0_all: Transition probability matrix for passive action (shape: [num_machines, num_states, num_states])
    - P1_all: Transition probability matrix for active action (shape: [num_machines, num_states, num_states])
    - R0_all: Reward matrix for passive action (shape: [num_machines, num_states])
    - R1_all: Reward matrix for active action (shape: [num_machines, num_states])

    Returns:
    - optimal_x: The optimal occupation measure (distribution over state-action pairs)
    - optimal_value: The optimal expected reward
    """

    num_machines, num_states, _ = P0_all.shape
    num_actions = 2  # Actions: 0 (passive), 1 (active)

    # Decision variables: x[s, a] represents the occupation measure for state-action pairs
    x = cp.Variable((num_states, num_actions), nonneg=True)

    # Constraints
    constraints = []

    # Flow conservation constraint: Ensures probability consistency
    for s in range(num_states):
        transition_sum = cp.sum(cp.multiply(P0_all[:, :, s], x[:, 0])) + cp.sum(cp.multiply(P1_all[:, :, s], x[:, 1]))
        constraints.append(transition_sum == cp.sum(x[s, :]))  # Ensures the flow balance

    # Normalization constraint: Ensures that total probability sums to 1
    constraints.append(cp.sum(x) == 1)

    # Define expected reward function
    reward = cp.sum(cp.multiply(x[:, 0], np.sum(R0_all, axis=0)) + cp.multiply(x[:, 1], np.sum(R1_all, axis=0)))

    # Solve LP to maximize reward
    problem = cp.Problem(cp.Maximize(reward), constraints)
    problem.solve()

    # Extract optimal values
    optimal_x = x.value
    optimal_value = problem.value

    return optimal_x, optimal_value


In [ ]:
import numpy as np

def generate_machine_states(num_damage_states=8, num_machines=3):
    """Generate all possible machine states"""
    from itertools import product
    return np.array(list(product(range(num_damage_states), repeat=num_machines)))

def initialize_transition_matrices():
    """
    Creates separate transition matrices P0 (no repair) and P1 (with repair) for all machines.
    """
    P0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))
    P1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        transitions = generate_transition_probabilities()

        # Fill P0 (No repair scenario) for machine m
        for i in range(NUM_DAMAGE_STATES - 1):
            P0_all[m, i, i + 1] = transitions[i, 0]  # Move to worse state
            P0_all[m, i, 0] = transitions[i, 1]  # Reset to pristine

        P0_all[m, -1, 0] = 1  # Most damaged state resets to pristine

        # Fill P1 (With repair scenario) for machine m
        P1_all[m, :, 0] = 1  # Repair resets any state to pristine

    return P0_all, P1_all

def initialize_reward_matrices(P0_all, D, a):
    """
    Initializes the reward matrices R0 (passive) and R1 (repair).
    """
    R0_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))
    R1_all = np.zeros((NUM_MACHINES, NUM_DAMAGE_STATES))

    for m in range(NUM_MACHINES):
        Cm = np.random.uniform(D, D + 25)
        Km = a * Cm

        # Passive Reward Matrix (R0)
        for k in range(NUM_DAMAGE_STATES):
            R0_all[m, k] = P0_all[m, k, 0] * (-Km)  # Penalty when transitioning to pristine

        # Repair Reward Matrix (R1)
        R1_all[m, :] = -Cm  # Repair incurs fixed cost

    return R0_all, R1_all

def whittle_indices(P0, P1, R0, R1):
  model = bandit.restless_bandit_from_P0P1_R0R1(P0, P1, R0, R1)
  whittle_indices = model.whittle_indices(discount = 0.95)
  return whittle_indices

def compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all, max_iterations=10, gamma=0.95, tol=1e-6):
    """Compute optimal and Whittle index costs using Value Iteration"""
    all_machine_states = generate_machine_states()
    state_to_index = {tuple(state): idx for idx, state in enumerate(all_machine_states)}
    num_states = len(all_machine_states)

    V = np.zeros(num_states)
    V_whittle = np.zeros(num_states)
    indices_all = np.zeros((P0_all.shape[0], P0_all.shape[1]))

    for m in range(P0_all.shape[0]):
        indices_all[m] = whittle_indices(P0_all[m], P1_all[m], R0_all[m], R1_all[m])

    for iteration in range(max_iterations):
        V_old = V.copy()
        V_whittle_old = V_whittle.copy()
        max_diff = 0

        for s, state in enumerate(all_machine_states):
            machine_indices = [indices_all[m, state[m]] for m in range(len(state))]

            print(f"Machine Indices = {machine_indices}")

            whittle_action = [1 if max(machine_indices) == i else 0 for i in machine_indices]
            print(f"Whittle Action : {whittle_action}")

            action_values = []  # Temporary storage for values of each action

            for action in [[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]:
                print(f"Action : {action}")
                R = [R0_all[m, state[m]] if action[m] == 0 else R1_all[m, state[m]] for m in range(len(state))]
                P = [P0_all[arm, state[arm]] if action[arm] == 0 else P1_all[arm, state[arm]] for arm in range(NUM_MACHINES)]

                """
                expected_value = sum(
                    P[0][ns1] * P[1][ns2] * P[2][ns3] * V_old[state_to_index[(ns1, ns2, ns3)]]
                    for ns1 in range(NUM_DAMAGE_STATES)
                    for ns2 in range(NUM_DAMAGE_STATES)
                    for ns3 in range(NUM_DAMAGE_STATES)
                )

                expected_whittle_value = sum(
                P[0][ns1] * P[1][ns2] * P[2][ns3] * V_whittle_old[state_to_index[(ns1, ns2, ns3)]]
                for ns1 in range(NUM_DAMAGE_STATES)
                for ns2 in range(NUM_DAMAGE_STATES)
                for ns3 in range(NUM_DAMAGE_STATES)
                )

                action_values.append(sum(R) + gamma * expected_value)  # temp list
                print(f"action_values : {action_values}")
                """


                expected_prob = sum(
                    P[0][ns1] * P[1][ns2] * P[2][ns3]
                    for ns1 in range(NUM_DAMAGE_STATES) for ns2 in range(NUM_DAMAGE_STATES) for ns3 in range(NUM_DAMAGE_STATES)
                )
                v_opt = sum(
                    P[0][ns1] * P[1][ns2] * P[2][ns3] * (sum(R) + gamma *  V_old[state_to_index[(ns1, ns2, ns3)]])
                    for ns1 in range(NUM_DAMAGE_STATES) for ns2 in range(NUM_DAMAGE_STATES) for ns3 in range(NUM_DAMAGE_STATES)
                )
                action_values.append(v_opt)

                v_whittle = sum(
                    P[0][ns1] * P[1][ns2] * P[2][ns3] * (sum(R) + gamma *  V_whittle_old[state_to_index[(ns1, ns2, ns3)]])
                    for ns1 in range(NUM_DAMAGE_STATES) for ns2 in range(NUM_DAMAGE_STATES) for ns3 in range(NUM_DAMAGE_STATES)
                )
                if action == whittle_action:
                    V_whittle[s] = v_whittle

                print(f"V_whittle[s] : {V_whittle[s]}")
            V[s] = np.clip(max(action_values), -1e6, 1e6)

            #(f"len of V[s] = {V[s].size} ")
            #print(f"len of V_whittle[s] = {V_whittle[s].size} ")

            state_diff = abs(V[s] - V_old[s]) + abs(V_whittle[s] - V_whittle_old[s])
            max_diff = max(max_diff, state_diff)

        if max_diff < tol:
          print(f"Converged after {iteration+1} iterations")
          break

    return V, V_whittle

def analyze_suboptimality(D, a, num_problems=10):
    """Compute the percentage cost suboptimality for multiple problem instances"""
    suboptimalities = []

    for _ in range(num_problems):
        P0_all, P1_all = initialize_transition_matrices()
        R0_all, R1_all = initialize_reward_matrices(P0_all, D, a)
        #optimal_x, optimal_value = solve_restless_bandits_lp(P0_all, P1_all, R0_all, R1_all)
        V_optimal, V_whittle = compute_optimal_and_whittle_cost(D, a, P0_all, P1_all, R0_all, R1_all)

        #print(f"Optimal Value using LP Program : {optimal_value}")
        print(f"V_whittle.shape = {V_whittle.shape}, V_optimal.shape = {V_optimal.shape}")
        print(f"V_optimal: {V_optimal.mean()}, V_whittle: {V_whittle.mean()}")
        #print("Suboptimality : ", 100 * (V_whittle - V_optimal) / V_optimal)
        suboptimalities.append(100 * (V_whittle.mean() - V_optimal.mean()) / V_optimal.mean())

    lower_quartile = np.percentile(suboptimalities, 25)
    median = np.percentile(suboptimalities, 50)
    upper_quartile = np.percentile(suboptimalities, 75)

    print(f"Suboptimality (Lower Quartile: {lower_quartile:.2f}, Median: {median:.2f}, Upper Quartile: {upper_quartile:.2f})")
    return lower_quartile, median, upper_quartile

In [ ]:
D, a = 0, 2.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
Action : [0, 0, 0]
action_values : [np.float64(-220.35444149378463)]
V_whittle[s] : -218.29898706344284
Action : [1, 0, 0]
action_values : [np.float64(-220.35444149378463), np.float64(-220.34628851439084)]
V_whittle[s] : -218.29898706344284
Action : [0, 1, 0]
action_values : [np.float64(-220.35444149378463), np.float64(-220.34628851439084), np.float64(-216.80182033202485)]
V_whittle[s] : -218.29898706344284
Action : [0, 0, 1]
action_values : [np.float64(-220.35444149378463), np.float64(-220.34628851439084), np.float64(-216.80182033202485), np.float64(-215.81835570388438)]
V_whittle[s] : -218.2989875588776
Machine Indices = [np.float64(-0.4076993641674125), np.float64(4.180501664628084), np.float64(5.85273295692383)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-221.08941066079433)]
V_whittle[s] : -218.29898706344284
Action : [1, 0, 0]
action_values : [np.float64(-221.08941066079433), np.float64(-221.1009979

(np.float64(1.2685425474806158),
 np.float64(2.3156608429538625),
 np.float64(3.9996342945547845))

In [ ]:
D, a = 20, 2.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
Action : [0, 0, 0]
action_values : [np.float64(-1244.8719664874402)]
V_whittle[s] : -1235.4303675599585
Action : [1, 0, 0]
action_values : [np.float64(-1244.8719664874402), np.float64(-1245.1513355512493)]
V_whittle[s] : -1235.4303675599585
Action : [0, 1, 0]
action_values : [np.float64(-1244.8719664874402), np.float64(-1245.1513355512493), np.float64(-1222.310224291087)]
V_whittle[s] : -1235.4303675599585
Action : [0, 0, 1]
action_values : [np.float64(-1244.8719664874402), np.float64(-1245.1513355512493), np.float64(-1222.310224291087), np.float64(-1220.3210413891309)]
V_whittle[s] : -1235.4303680515006
Machine Indices = [np.float64(-2.526291106525366), np.float64(24.15779581998578), np.float64(33.756029327591676)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-1252.7352429252783)]
V_whittle[s] : -1235.4303675599585
Action : [1, 0, 0]
action_values : [np.float64(-1252.7352429252783), np.float64(-1253.489813

(np.float64(2.0798024370094055),
 np.float64(3.8634978105858195),
 np.float64(5.124776420212864))

In [ ]:
D, a = 10, 2.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
Action : [0, 0, 0]
action_values : [np.float64(-713.9668017451081)]
V_whittle[s] : -746.6724112442303
Action : [1, 0, 0]
action_values : [np.float64(-713.9668017451081), np.float64(-711.7537294334057)]
V_whittle[s] : -746.6724112442303
Action : [0, 1, 0]
action_values : [np.float64(-713.9668017451081), np.float64(-711.7537294334057), np.float64(-707.747619698176)]
V_whittle[s] : -746.6724112442303
Action : [0, 0, 1]
action_values : [np.float64(-713.9668017451081), np.float64(-711.7537294334057), np.float64(-707.747619698176), np.float64(-706.1155364858283)]
V_whittle[s] : -746.6724117378546
Machine Indices = [np.float64(1.7604672461771944), np.float64(5.789480514981717), np.float64(14.011447965155716)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-718.0818086759986)]
V_whittle[s] : -746.6724112442303
Action : [1, 0, 0]
action_values : [np.float64(-718.0818086759986), np.float64(-716.029536261113)]
V_whittle

(np.float64(2.994571449001047),
 np.float64(5.217407190113741),
 np.float64(6.3242937950415214))

In [ ]:
D, a = 0, 5.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
Action : [0, 0, 0]
action_values : [np.float64(-863.9404559435726)]
V_whittle[s] : -817.9149342339564
Action : [1, 0, 0]
action_values : [np.float64(-863.9404559435726), np.float64(-817.9149347098117)]
V_whittle[s] : -817.9149347229127
Action : [0, 1, 0]
action_values : [np.float64(-863.9404559435726), np.float64(-817.9149347098117), np.float64(-843.7162107118113)]
V_whittle[s] : -817.9149347229127
Action : [0, 0, 1]
action_values : [np.float64(-863.9404559435726), np.float64(-817.9149347098117), np.float64(-843.7162107118113), np.float64(-836.4037515314865)]
V_whittle[s] : -817.9149347229127
Machine Indices = [np.float64(46.21619525161972), np.float64(15.477223425774051), np.float64(53.76399104313625)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-881.6600890718623)]
V_whittle[s] : -836.4037510556311
Action : [1, 0, 0]
action_values : [np.float64(-881.6600890718623), np.float64(-838.0015596517927)]
V_whitt

(np.float64(0.017596403626459246),
 np.float64(0.0426422559068808),
 np.float64(0.0512848458978401))

In [ ]:
D, a = 10, 5.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
V_whittle[s] : -943.6085446167378
Action : [0, 1, 0]
action_values : [np.float64(-999.0685743103681), np.float64(-999.7285755306415), np.float64(-943.7305532544565)]
V_whittle[s] : -943.6085446167378
Action : [0, 0, 1]
action_values : [np.float64(-999.0685743103681), np.float64(-999.7285755306415), np.float64(-943.7305532544565), np.float64(-943.6429459402318)]
V_whittle[s] : -943.6443966628967
Machine Indices = [np.float64(-6.508675815944002), np.float64(58.2157111938629), np.float64(131.2408810892141)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-1065.308029158756)]
V_whittle[s] : -943.6085446167378
Action : [1, 0, 0]
action_values : [np.float64(-1065.308029158756), np.float64(-1068.8219386604414)]
V_whittle[s] : -943.6085446167378
Action : [0, 1, 0]
action_values : [np.float64(-1065.308029158756), np.float64(-1068.8219386604414), np.float64(-1014.102102169874)]
V_whittle[s] : -943.6085446167378
Action :

In [ ]:
D, a = 20, 5.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
Whittle Action : [0, 1, 0]
Action : [0, 0, 0]
action_values : [np.float64(-1349.858570334736)]
V_whittle[s] : -1271.6306394571172
Action : [1, 0, 0]
action_values : [np.float64(-1349.858570334736), np.float64(-1364.9804312608012)]
V_whittle[s] : -1271.6306394571172
Action : [0, 1, 0]
action_values : [np.float64(-1349.858570334736), np.float64(-1364.9804312608012), np.float64(-1256.6966552481863)]
V_whittle[s] : -1271.6331572259644
Action : [0, 0, 1]
action_values : [np.float64(-1349.858570334736), np.float64(-1364.9804312608012), np.float64(-1256.6966552481863), np.float64(-1246.8914044523628)]
V_whittle[s] : -1271.6331572259644
Machine Indices = [np.float64(-21.066422938212703), np.float64(124.40792642270493), np.float64(138.35376241192222)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
action_values : [np.float64(-1370.2457973926826)]
V_whittle[s] : -1261.9317987015681
Action : [1, 0, 0]
action_values : [np.float64(-1370.245797392682

In [ ]:
D, a = 25, 5.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
V_whittle[s] : -890.5525051068537
Action : [1, 0, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 1, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 0, 1]
V_whittle[s] : -961.726232289936
Machine Indices = [np.float64(-6.90319756092854), np.float64(78.62839655905825), np.float64(167.5811630807917)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
V_whittle[s] : -890.5525051068537
Action : [1, 0, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 1, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 0, 1]
V_whittle[s] : -961.726232289936
Machine Indices = [np.float64(-6.90319756092854), np.float64(78.62839655905825), np.float64(171.07878978972573)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
V_whittle[s] : -890.5525051068537
Action : [1, 0, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 1, 0]
V_whittle[s] : -890.5525051068537
Action : [0, 0, 1]
V_whittle[s] : -961.726232289936
Machine Indices = [np.float64(-6.90319756092854), np.flo

(np.float64(0.41087669970817775),
 np.float64(0.5674324090197811),
 np.float64(0.7889049691519514))

In [ ]:
D, a = 30, 5.0
analyze_suboptimality(D, a)

Streaming output truncated to the last 5000 lines.
V_whittle[s] : -645.3425885961312
Action : [1, 0, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 1, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 0, 1]
V_whittle[s] : -698.2720026255097
Machine Indices = [np.float64(-17.312206332861134), np.float64(38.69884264540995), np.float64(126.56076956239761)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
V_whittle[s] : -645.3425885961312
Action : [1, 0, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 1, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 0, 1]
V_whittle[s] : -698.2720026255097
Machine Indices = [np.float64(-17.312206332861134), np.float64(38.69884264540995), np.float64(134.26903249586857)]
Whittle Action : [0, 0, 1]
Action : [0, 0, 0]
V_whittle[s] : -645.3425885961312
Action : [1, 0, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 1, 0]
V_whittle[s] : -645.3425885961312
Action : [0, 0, 1]
V_whittle[s] : -698.2720026255097
Machine Indices = [np.float64(-17.31220633286113

(np.float64(0.6059647354603502),
 np.float64(0.7686790118852712),
 np.float64(0.9106793225119008))